# Desafio Spacecom: análise exploratória dos dados

> Exploração dos datasets de posicionamento antes de implementar o algoritmo de detecção de encontros.  
> O objetivo é entender o formato, a qualidade e o perfil dos dados que o algoritmo vai processar.

## 1. Configuração

In [59]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

N = 1


def achar_pasta(nome):
    for candidata in (Path(nome), Path("..") / nome):
        if candidata.is_dir():
            return candidata


DATA_DIR = achar_pasta("data")
OUTPUT_DIR = achar_pasta("output")
print("datasets em:", DATA_DIR.resolve())

datasets em: /Users/gabrielastdossantos/desafio-trainee-spacecom/data


## 2. Carregamento do dataset

In [60]:
def carregar(n):
    df = pd.read_csv(
        DATA_DIR / f"dataset{n}.txt",
        sep=";",
        dtype={"faccaoMonitorado": "string", "idsRestricaoMedida": "string"},
    )
    df["timestamp"] = pd.to_datetime(df["timestamp"], format="%Y-%m-%d %H:%M:%S")
    df["medidaProtetiva"] = df["medidaProtetiva"].astype(str).str.lower().eq("true")
    return df


df = carregar(N)
print("linhas x colunas:", df.shape)
df.head(10)

linhas x colunas: (5000, 7)


,idMonitorado,faccaoMonitorado,timestamp,latitude,longitude,medidaProtetiva,idsRestricaoMedida
0,1,<NA>,2024-01-01,-25.59386,-73.96319,False,<NA>
1,2,<NA>,2024-01-01,-21.15463,-61.43519,False,<NA>
2,3,<NA>,2024-01-01,-28.21180,-44.64074,False,<NA>
3,4,<NA>,2024-01-01,-31.94294,-40.63330,False,<NA>
4,5,<NA>,2024-01-01,-28.28862,-44.57253,False,<NA>
5,6,<NA>,2024-01-01,-31.92886,-40.61590,False,<NA>
6,7,1,2024-01-01,-13.06443,-46.56289,False,<NA>
7,8,1,2024-01-01,-13.16420,-46.79759,False,<NA>
8,9,<NA>,2024-01-01,1.64046,-69.53947,True,10
9,10,<NA>,2024-01-01,1.85060,-69.58072,False,<NA>


In [61]:
df.dtypes

idMonitorado                   int64
faccaoMonitorado              string
timestamp             datetime64[us]
latitude                     float64
longitude                    float64
medidaProtetiva                 bool
idsRestricaoMedida            string
dtype: object

## 3. Qualidade dos dados

In [62]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 7 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   idMonitorado        5000 non-null   int64         
 1   faccaoMonitorado    1000 non-null   string        
 2   timestamp           5000 non-null   datetime64[us]
 3   latitude            5000 non-null   float64       
 4   longitude           5000 non-null   float64       
 5   medidaProtetiva     5000 non-null   bool          
 6   idsRestricaoMedida  500 non-null    string        
dtypes: bool(1), datetime64[us](1), float64(2), int64(1), string(2)
memory usage: 239.4 KB


In [63]:
df.describe()

,idMonitorado,timestamp,latitude,longitude
count,5000.000000,5000,5000.000000,5000.000000
mean,5.500000,2024-01-01 00:20:47.500000,-18.949990,-53.824037
min,1.000000,2024-01-01 00:00:00,-31.952740,-73.989970
25%,3.000000,2024-01-01 00:10:23.750000,-28.353030,-69.537802
50%,5.500000,2024-01-01 00:20:47.500000,-23.246090,-46.500640
75%,8.000000,2024-01-01 00:31:11.250000,-13.029470,-44.624707
max,10.000000,2024-01-01 00:41:35,2.119820,-40.483430
std,2.872569,NaN,12.223935,12.596858


In [64]:
print("Nulos por coluna:")
print(df.isna().sum())

dup = df.duplicated(["idMonitorado", "timestamp"]).sum()
print("\nlinhas repetidas para o mesmo (id, timestamp):", dup)

Nulos por coluna:
idMonitorado             0
faccaoMonitorado      4000
timestamp                0
latitude                 0
longitude                0
medidaProtetiva          0
idsRestricaoMedida    4500
dtype: int64

linhas repetidas para o mesmo (id, timestamp): 0


In [65]:
ts = df["timestamp"].drop_duplicates().sort_values()
passos = ts.diff().dropna().value_counts()
print("Primeiro timestamp:", ts.iloc[0])
print("Último timestamp:  ", ts.iloc[-1])
print("Quantidade de timestamps distintos:", len(ts))
print("\nIntervalos entre timestamps consecutivos:")
print(passos)

Primeiro timestamp: 2024-01-01 00:00:00
Último timestamp:   2024-01-01 00:41:35
Quantidade de timestamps distintos: 500

Intervalos entre timestamps consecutivos:
timestamp
0 days 00:00:05    499
Name: count, dtype: int64


In [66]:
total_ids = df["idMonitorado"].nunique()
por_instante = df.groupby("timestamp")["idMonitorado"].nunique()
print("ids distintos no dataset:", total_ids)
print("\nids presentes por timestamp:")
print(por_instante.describe())
print("\nTodo id aparece em todo timestamp?", bool((por_instante == total_ids).all()))

ids distintos no dataset: 10

ids presentes por timestamp:
count    500.0
mean      10.0
std        0.0
min       10.0
25%       10.0
50%       10.0
75%       10.0
max       10.0
Name: idMonitorado, dtype: float64

Todo id aparece em todo timestamp? True


## 4. Perfil das pessoas monitoradas

In [67]:
pessoas = df.groupby("idMonitorado").agg(
    faccao=("faccaoMonitorado", "first"),
    vitima=("medidaProtetiva", "max"),
    n_posicoes=("timestamp", "count"),
)

print("Vítima (True) x tornozeleira (False):")
print(pessoas["vitima"].value_counts())
print("\nPessoas por facção (<NA> = sem facção):")
print(pessoas["faccao"].value_counts(dropna=False))

Vítima (True) x tornozeleira (False):
vitima
False    9
True     1
Name: count, dtype: int64

Pessoas por facção (<NA> = sem facção):
faccao
<NA>    8
1       2
Name: count, dtype: Int64


In [68]:
# Facção ou flag de vítima mudam para a mesma pessoa?
muda_faccao = df.groupby("idMonitorado")["faccaoMonitorado"].nunique(dropna=False).gt(1).sum()
muda_vitima = df.groupby("idMonitorado")["medidaProtetiva"].nunique().gt(1).sum()
print("pessoas cuja facção muda:", muda_faccao)
print("pessoas cujo flag de vítima muda:", muda_vitima)

# Vítimas com facção preenchida
print("\nvítimas com facção:", int((pessoas["vitima"] & pessoas["faccao"].notna()).sum()))

pessoas cuja facção muda: 0
pessoas cujo flag de vítima muda: 0

vítimas com facção: 0
